# qobuzdl-collab — Google Colab

Download albums / tracks / full artist discographies from Qobuz.

**Defaults**
- Folder: `{album} - {year} [{quality}]`
- Track: `{title}`
- Quality: hi-res-192 with fallback

Repo: https://github.com/zenin-373/qobuzdl-collab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenin-373/qobuzdl-collab/blob/main/qobuzdl_collab_colab.ipynb)

## 1. Install
Run once per session.

In [ ]:
from pathlib import Path
import shutil

WORK = Path("/content/qobuzdl-collab")
if WORK.exists():
    shutil.rmtree(WORK)

!pip install -q click requests rich mutagen
!git clone -q https://github.com/zenin-373/qobuzdl-collab.git /content/qobuzdl-collab
%cd /content/qobuzdl-collab
!curl -s -o qobuz_dl/downloader.py https://raw.githubusercontent.com/jcomicsutils/qobuz-dl/main/qobuz_dl/downloader.py
!pip install -q -e .
print("Installed at /content/qobuzdl-collab")

## 2. Config (form)
Fill the boxes, then **run this cell**. No interactive setup needed.

- Tokens: comma-separated if multiple
- Use the app_id/secret your token checker shows for that token version
- Track template must NOT include `{quality}` (only folder can)

In [ ]:
# @title Qobuz credentials and defaults
# @markdown ### Credentials
app_id = "798273057"  # @param {type:"string"}
secret = "abb21364945c0583309667d13ca3d93a"  # @param {type:"string"}
auth_tokens = "PASTE_TOKEN_HERE"  # @param {type:"string"}
# @markdown One token, or several separated by commas.

# @markdown ### Paths
download_dir = "/content/drive/Shareddrives/Asa-Mikata/Qobuz DL"  # @param {type:"string"}

# @markdown ### Quality
quality = "hi-res-192"  # @param ["hi-res-192", "hi-res", "cd", "mp3"]

# @markdown ### Naming
# @markdown Folder example: Thriller - 1982 [FLAC 24bit 176kHz]
folder_template = "{album} - {year} [{quality}]"  # @param {type:"string"}
track_template = "{title}"  # @param {type:"string"}

# @markdown ### Options
quality_fallback = True  # @param {type:"boolean"}
duration_check = True  # @param {type:"boolean"}
save_cover = True  # @param {type:"boolean"}
embed_metadata = True  # @param {type:"boolean"}
skip_existing = True  # @param {type:"boolean"}
multi_disc = True  # @param {type:"boolean"}

import json
from pathlib import Path

tokens = []
for part in auth_tokens.replace("\n", ",").split(","):
    t = part.strip()
    if t and t != "PASTE_TOKEN_HERE":
        tokens.append(t)

config = {
    "app_id": app_id.strip(),
    "secret": secret.strip(),
    "auth_tokens": tokens,
    "download_dir": download_dir.strip(),
    "quality": quality,
    "folder_template": folder_template.strip(),
    "track_template": track_template.strip(),
    "quality_fallback": quality_fallback,
    "quality_fallback_path": ["hi-res-192", "hi-res", "cd"],
    "duration_check": duration_check,
    "save_cover": save_cover,
    "embed_metadata": embed_metadata,
    "force_main_album_artist": True,
    "skip_existing": skip_existing,
    "retries": 3,
    "multi_disc": multi_disc,
    "on_final_failure": "delete_partial",
    "include_version": True,
    "strip_feat_from_album_title": False,
    "strip_feat_from_track_title": False,
    "cover_size": "original",
    "embed_cover_size": "original",
    "embed_cover_oversize_action": "use_large",
}

cfg_dir = Path("/root/.config/qobuz-dl")
cfg_dir.mkdir(parents=True, exist_ok=True)
cfg_path = cfg_dir / "config.json"
cfg_path.write_text(json.dumps(config, indent=2))

print("Config saved →", cfg_path)
print("app_id:", config["app_id"])
print("tokens:", len(config["auth_tokens"]))
print("download_dir:", config["download_dir"])
print("folder:", config["folder_template"])
print("track:", config["track_template"])
if not config["auth_tokens"]:
    print("WARNING: paste at least one auth token, then re-run")
else:
    print("Ready")

## 3. Auth test (optional)
Expect Status: 200.

In [ ]:
import json, requests
from pathlib import Path

cfg = json.loads(Path("/root/.config/qobuz-dl/config.json").read_text())
r = requests.get(
    "https://www.qobuz.com/api.json/0.2/user/get",
    headers={
        "X-App-Id": cfg["app_id"],
        "X-User-Auth-Token": cfg["auth_tokens"][0],
    },
    timeout=20,
)
print("Status:", r.status_code)
print(r.text[:400])

## 4. Download

In [ ]:
# @title Download
%cd /content/qobuzdl-collab

id_type = "al-id"  # @param ["ar-id", "al-id", "tr-id"]
id_number = "0074643811224"  # @param {type:"string"}

!qobuz-dl dl {id_type} {id_number}

## 5. Zip (optional)

In [ ]:
# @title Zip
zip_filename = "/content/Qobuz_Downloads.zip"  # @param {type:"string"}
source_dir = "/content/drive/Shareddrives/Asa-Mikata/Qobuz DL"  # @param {type:"string"}

import os
from google.colab import files

!zip -r "{zip_filename}" "{source_dir}"
if os.path.exists(zip_filename):
    files.download(zip_filename)
else:
    print("Zip failed or path empty")